# Parallel Architecture — LangChain

**Pattern:** Parallel (fan-out → independent research → aggregate)  
**Framework:** LangChain LCEL + asyncio  
**Task:** 3 city researchers run simultaneously → aggregator ranks them

## Architecture

```
               Input: [Tokyo, Paris, Bangalore]
                          │
          asyncio.gather()─┼──────────────────┐
          │                │                  │
          ▼                ▼                  ▼
   researcher(Tokyo) researcher(Paris) researcher(Bangalore)
          │                │                  │
          └────────────────┼──────────────────┘
                           ▼
                    aggregator_chain
                           │
                        Ranking
```

**Key concept:** `asyncio.gather()` fires all chain calls simultaneously.  
Each call is independent — they share no state and don't wait for each other.

In [ ]:
import asyncio
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
parser = StrOutputParser()
print("✓ Ready")

In [ ]:
CITY_DATA = {
    "tokyo":     "Weather: Clear, 18°C, score 9/10. Safety: Low, score 10/10. Time: 22:30 JST.",
    "paris":     "Weather: Partly Cloudy, 16°C, score 7/10. Safety: Low, score 8/10. Time: 15:30 CET.",
    "bangalore": "Weather: Rainy, 25°C, score 6/10. Safety: Medium, score 6/10. Time: 20:00 IST.",
}

researcher_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Write a structured 3-line city travel report: Weather, Safety, Recommendation."),
        ("human", "City: {city}\nData: {raw_data}"),
    ]) | llm | parser
)

aggregator_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Rank these cities best to worst by weather + safety. Justify each rank."),
        ("human", "Reports:\n\n{city_reports}\n\nRank and name the top pick."),
    ]) | llm | parser
)

print("Chains ready")

In [ ]:
import time

async def research_city_async(city: str) -> str:
    raw = CITY_DATA.get(city.lower(), "No data.")
    result = await researcher_chain.ainvoke({"city": city, "raw_data": raw})
    print(f"  [parallel] {city} done")
    return f"### {city}\n{result}"

# Time the parallel execution
start = time.time()
reports = await asyncio.gather(
    research_city_async("Tokyo"),
    research_city_async("Paris"),
    research_city_async("Bangalore"),
)
elapsed = time.time() - start
print(f"\n3 cities researched in {elapsed:.1f}s (parallel)")

for r in reports:
    print("\n" + r)

In [ ]:
# Aggregate all reports into a final ranking
combined = "\n\n".join(reports)
ranking = aggregator_chain.invoke({"city_reports": combined})
print("## Final Ranking")
print(ranking)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `asyncio.gather()` | Standard Python concurrency — no framework needed for parallel LangChain calls |
| `.ainvoke()` vs `.invoke()` | Async chain call — same chain, different entry point |
| Aggregator receives all reports | Single aggregator reduces N reports to 1 ranking |
| Same chain for all cities | Fan-out = same logic, different inputs |

### Parallel vs Sequential for this task
- Sequential: 3 LLM calls in series → ~3× slower
- Parallel: 3 LLM calls at once → latency of 1 call

### Next: [LangGraph Parallel →](../LangGraph/parallel.ipynb)